# M0 오리엔테이션 — 실습: Colab 첫걸음 & 첫 실험 (W1)

> 위에서부터 한 셀씩 `Shift+Enter`로 실행하세요. `___` 빈칸은 직접 채웁니다.

**이 실습이 끝나면**
1. Colab에서 코드 셀을 실행할 수 있다
2. 붓꽃(iris) 데이터를 불러와 살펴본다 (150×5, 품종 50/50/50)
3. **세트피스:** 손 규칙 두 개를 검산한다 — 규칙 1은 **1.0**, 규칙 2는 **0.93**에서 멈춤 = "학습"의 이유 ⭐
4. '맛보기' 학습 모델(KNN)을 한 번 돌려본다 (원리는 M3에서)

**7단계 멘탈모델:** 오늘은 데이터→…→활용 *전체 흐름*을 한 바퀴 돕니다.

## 0. Colab 사용법 빠른 체험
아래 셀을 클릭하고 `Shift+Enter`를 눌러 실행해 보세요.

In [ ]:
print('안녕하세요, 기계학습!')               # 화면에 문자열 출력
import sys                                    # 시스템 정보 모듈
print('파이썬 버전:', sys.version.split()[0]) # 파이썬 버전만 출력

## 1. 라이브러리 불러오기
이 과목에서 자주 쓰는 도구들입니다. (Colab에 이미 설치되어 있음)

In [ ]:
import numpy as np                       # 수치 배열 계산 도구
import pandas as pd                      # 표(데이터프레임) 도구
import matplotlib.pyplot as plt          # 그래프 도구
from sklearn.datasets import load_iris   # 붓꽃 예제 데이터 로더

print('준비 완료!')                       # 임포트 확인

## 2. 첫 데이터셋 — 붓꽃(iris)
붓꽃 3종을 꽃받침/꽃잎의 길이·너비 4개 특징으로 구분하는 유명한 입문 데이터입니다(1936년부터!).

**7단계 중 '데이터' 단계.**

In [ ]:
iris = load_iris(as_frame=True)   # 붓꽃 데이터를 표 형태로 로드
df = iris.frame                    # 특징 4개 + 정답(target)이 담긴 표
df.head()                          # 앞 5행 미리보기

## 3. 데이터 살펴보기

In [ ]:
print('행 x 열:', df.shape)                           # (150, 5)
print('정답(품종) 이름:', iris.target_names.tolist())  # 0,1,2가 가리키는 품종
print('품종별 개수:')
print(df['target'].value_counts())                    # 50 / 50 / 50
df.describe()                                         # 숫자 열 요약 통계

## 4. 세트피스 — 규칙을 손으로 적어 본다 ⭐
"규칙을 학습시킨다"가 왜 필요한지, **직접 규칙을 적어** 확인합니다. 무기는 꽃잎 길이 하나.
reading §5의 실측(setosa 최대 1.9 vs 나머지 최소 3.0 — 틈 / versicolor~virginica는 4.5~5.1 겹침)을 코드로 검산하세요.

> 팁 두 가지: 파이썬에서 True=1, False=0이라 **비교 결과의 평균이 곧 정확도**가 됩니다. `np.where(조건, 1, 2)`는 조건이 참이면 1, 아니면 2를 주는 함수입니다.

In [ ]:
pl = df['petal length (cm)']                       # 무기: 꽃잎 길이 하나
print('setosa 최대:', pl[df['target'] == 0].max())  # 1.9
print('나머지 최소:', pl[df['target'] != 0].min())  # 3.0 — 깨끗한 틈 1.1cm!

rule1 = (pl < ___)                                 # ✍️ 빈칸: 실험 1의 문턱 — 틈 사이의 그 수
acc1 = (rule1 == (df['target'] == 0)).mean()       # 규칙 판정 vs 정답 비교
print('규칙 1(setosa) 정확도:', acc1)               # 1.0 — 150송이 전부 정답!

sub = df[df['target'] != 0]                        # 실험 2: versicolor(1) vs virginica(2)
pred = np.where(sub['petal length (cm)'] < ___, 1, 2)  # ✍️ 빈칸: 최선의 문턱(본문의 그 수)
acc2 = (pred == sub['target']).mean()
print('규칙 2 정확도:', acc2)                       # 0.93 — 겹침 구간의 7송이는 어쩔 수 없음

> **검산 포인트:** 규칙 1 = **1.0**(깨끗한 틈 — 규칙 코딩 성공), 규칙 2 = **0.93**(4.5~5.1 겹침 — 어떤 문턱도 7송이 실패). **현실 대부분은 실험 2** → "데이터에게 규칙을 찾게 하자 = 학습". 그리고 우리는 지금 **숫자로** 판단했습니다(평가 습관 — M2 예고).

## 5. 시각화 — 두 특징으로 산점도
꽃잎 길이·너비만으로도 품종이 꽤 나뉘는 걸 눈으로 봅니다.

In [ ]:
plt.figure(figsize=(6, 4))                                   # 그림 크기
plt.scatter(df['petal length (cm)'], df['petal width (cm)'],  # x=꽃잎길이, y=꽃잎너비
            c=df['target'], cmap='viridis')                   # 색 = 품종
plt.xlabel('petal length (cm)')                               # 축(영어)
plt.ylabel('petal width (cm)')
plt.title('Iris: petal length vs width')                      # 제목(영어)
plt.colorbar(label='target')                                  # 색 범례
plt.show()

### ✍️ 직접 해보기 — 꽃받침(sepal)으로 그리기
이번엔 **꽃받침** 길이·너비로 산점도를 그려 보세요. `___` 두 곳을 채우면 됩니다.

In [ ]:
plt.figure(figsize=(6, 4))                                      # 그림 크기
plt.scatter(df[___], df[___], c=df['target'], cmap='viridis')   # ✍️ 빈칸 두 곳: sepal 길이/너비 컬럼명
plt.xlabel('sepal length (cm)')                                # 축(영어)
plt.ylabel('sepal width (cm)')
plt.title('Iris: sepal length vs width')                       # 제목(영어)
plt.show()

# 관찰: 꽃받침으로는 꽃잎만큼 깔끔히 안 나뉨 — '어떤 특징을 쓰는가'가 중요(=표현, M1·M8b로 이어짐)

## 6. 맛보기 — 첫 '학습하는' 분류기
손 규칙(문턱 조정 = 사람)을 넘어, **손실·최적화를 기계에게 맡기는** 첫 경험입니다. 원리는 M3(KNN)에서 — 지금은 7단계 전체 흐름 체험만.

In [ ]:
from sklearn.model_selection import train_test_split   # 학습/시험 분리
from sklearn.neighbors import KNeighborsClassifier      # KNN 분류기

X = iris.data        # 입력 특징 4개(손 규칙은 1개만 썼음!)
y = iris.target      # 정답(품종)

X_train, X_test, y_train, y_test = train_test_split(    # 두 묶음으로 분리
    X, y, test_size=0.3, random_state=42)               # 30%는 시험용

model = KNeighborsClassifier(n_neighbors=3)   # 이웃 3개를 참고하는 KNN
model.fit(X_train, y_train)                   # 학습용 데이터로 학습

acc = model.score(___, ___)                   # ✍️ 빈칸 두 곳: '시험용' X와 y로 정확도
print(f'시험 정확도: {acc:.2%}')               # 100.00% — 네 특징을 쓰는 학습 모델

> **관찰:** 시험 정확도 **100%** — 손 규칙 2가 0.93에서 멈춘 문제를, 네 특징을 모두 쓰는 학습 모델이 넘습니다. 왜 train/test로 나눴는지(공정한 시험)는 M2에서, KNN의 원리는 M3에서 정면으로 다룹니다.

## 7. 🤖 AI 코파일럿 활용 (선택) — ai-native v1
막히면 AI 튜터에게 묻되, **먼저 스스로 생각**하고 답을 **실행으로 검증**하세요.

**좋은 질문 예시**
- "손 규칙 실험 1과 2의 결과 차이(1.0 vs 0.93)를 내가 설명해 볼 테니 허점을 찔러 줘."
- "`train_test_split`이 왜 필요한지 초보자에게 설명해 줘."
- "이 에러가 무슨 뜻이야? `[에러 붙여넣기]`"

**가드레일**
1. 먼저 손으로 생각 → 그 다음 AI
2. AI 코드는 *왜 그런지* 설명할 수 있을 때만 사용
3. AI 출력은 실행해서 결과로 검증

## 8. 정리 & 자가 점검

**오늘 한 일 3줄**
1. Colab에서 셀을 실행하고 붓꽃 150송이를 살펴봤다
2. **손 규칙 두 개를 검산**했다 — 깨끗한 틈은 1.0, 겹치는 경계는 **0.93 한계** = "학습"의 이유
3. 첫 학습 모델(KNN)로 7단계 흐름을 한 바퀴 돌았다(시험 정확도 100%)

**스스로 점검**
- [ ] `df.shape`가 무엇을 뜻하는지 안다
- [ ] 규칙 1과 규칙 2의 결과 차이를 숫자로 말할 수 있다
- [ ] 왜 train/test로 나눴는지 한 문장으로 말할 수 있다(자세한 건 M2)

**🔹심화 (선택)**
- `n_neighbors`를 1, 5, 15로 바꿔 정확도를 비교해 보세요(이 데이터는 쉬워서 전부 100%가 나옵니다 — "쉬운 데이터"라는 것도 정보!).
- 규칙 2의 문턱을 4.7, 4.9, 5.0으로 바꿔 정확도가 어떻게 변하는지 보세요(최선이 0.93 부근임을 확인).
- 꽃잎 **너비**(petal width) 하나로 실험 1·2를 다시 해 보세요 — 더 좋은 무기인가요?

**다음 시간(M1):** 표를 다루는 손 — numpy·pandas, 다섯 승객의 미니 타이타닉 손계산.